# Design Patterns in Python — Expert Interview Guide

Covers: Creational, Structural, Behavioral patterns implemented Pythonically.

> **Python difference:** Many GoF patterns are simpler or unnecessary in Python due to first-class functions, duck typing, and built-in features.

## 1. Singleton

Ensures only one instance of a class exists.

In [ ]:
import threading

# Method 1: Metaclass (most Pythonic for class-based)
class SingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        with cls._lock:
            if cls not in cls._instances:
                cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

# Method 2: Module-level (simplest — Python modules are singletons)
# config.py:
# _config = None
# def get_config():
#     global _config
#     if _config is None: _config = Config()
#     return _config

# Method 3: Decorator
def singleton(cls):
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

**Interview Insight:** In Python, modules are singletons — importing the same module from different files gives the same module object. For simple singletons, a module-level instance is cleaner than metaclass overhead.

## 2. Factory & Abstract Factory

In [ ]:
from abc import ABC, abstractmethod

# Factory Method
class Notification(ABC):
    @abstractmethod
    def send(self, message: str) -> str: ...

# Factory function (Pythonic)
def create_notification(channel: str) -> Notification:
    channels = {
        "email": EmailNotification,
        "sms": SMSNotification,
        "push": PushNotification,
    }
    cls = channels.get(channel.lower())
    if not cls:
        raise ValueError(f"Unknown channel: {channel}")
    return cls()

**Interview Insight:** In Python, factory functions (returning instances from a dict/registry) are often cleaner than factory classes. Use `cls()` from a registry dict — it's more readable than a chain of `if/elif`.

## 3. Builder — Fluent Interface

In [ ]:
class QueryBuilder:
    def __init__(self):
        self._table = ""
        self._conditions = []

    def from_table(self, table: str) -> "QueryBuilder":
        self._table = table
        return self

    def select(self, *columns) -> "QueryBuilder":
        self._columns = list(columns)
        return self

    def where(self, condition: str) -> "QueryBuilder":
        self._conditions.append(condition)
        return self

    def build(self) -> str:
        # Build SQL query
        return f"SELECT ... FROM {self._table}"

# Usage with method chaining
query = (
    QueryBuilder()
    .from_table("orders")
    .select("id", "customer", "total")
    .where("total > 100")
    .build()
)

**Interview Insight:** The Builder pattern in Python uses method chaining (`return self`). Use `Self` type hint (Python 3.11+) to preserve the subclass type in inherited builders.

## 4. Observer — Event System

In [ ]:
class Event:
    def __init__(self, name: str):
        self.name = name
        self._handlers = []

    def subscribe(self, handler):
        self._handlers.append(handler)

    def fire(self, *args, **kwargs):
        for handler in self._handlers:
            handler(*args, **kwargs)

    def __iadd__(self, handler):
        self.subscribe(handler)
        return self

**Interview Insight:** Use `weakref` for observers attached to UI components or short-lived objects. Without weakref, the event system holds a strong reference, preventing garbage collection of dead subscribers.

## 5. Strategy Pattern

In [ ]:
from typing import Protocol, Callable

# Pythonic strategy: just pass a function!
def sort_by_name(items): return sorted(items, key=lambda x: x["name"])
def sort_by_price(items): return sorted(items, key=lambda x: x["price"])

class ProductList:
    def __init__(self, products):
        self.products = products

    def display(self, sort_strategy: Callable = sort_by_name):
        sorted_items = sort_strategy(self.products)
        for p in sorted_items:
            print(f"  {p['name']:15} ${p['price']:.2f}")

**Interview Insight:** In Python, the Strategy pattern reduces to passing a function. Classes are only needed when strategies have state. Use `Protocol` for type-safe strategy interfaces without inheritance.

## 6. Context Manager Pattern

In [ ]:
from contextlib import contextmanager
import time

# Class-based context manager
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False  # don't suppress exceptions

# Generator-based context manager
@contextmanager
def transaction(db_name):
    print(f"[TX] BEGIN {db_name}")
    try:
        yield {"name": db_name, "active": True}
        print(f"[TX] COMMIT {db_name}")
    except Exception as e:
        print(f"[TX] ROLLBACK {db_name}: {e}")
        raise

**Interview Insight:** `__exit__` returning `True` suppresses the exception. Return `False` (or `None`) to propagate it. The `contextlib.suppress(ExcType)` is equivalent to a try/except that swallows the exception.

## 7. Command Pattern — Undo/Redo

In [ ]:
from abc import ABC, abstractmethod
from collections import deque

class Command(ABC):
    @abstractmethod
    def execute(self) -> None: ...
    @abstractmethod
    def undo(self) -> None: ...

class TextEditor:
    def __init__(self):
        self.text = ""
        self._history = deque()
        self._redo_stack = deque()

    def execute(self, cmd: Command) -> None:
        cmd.execute()
        self._history.append(cmd)
        self._redo_stack.clear()

    def undo(self) -> None:
        if not self._history: return
        cmd = self._history.pop()
        cmd.undo()
        self._redo_stack.append(cmd)

**Interview Insight:** The Command pattern encapsulates an operation as an object, enabling undo/redo, queuing, logging, and retry. In Python, simple commands can be functions in a list rather than full Command objects.

See 07_design_patterns.ipynb for full implementation with working examples and demonstrations.